In [37]:
from pathlib import Path
import jsonlines
import pandas as pd
import matplotlib.pyplot as plt

models = ['mmbert']
experiments = ['baseline', 'exp_1_mecla', 'exp_6_mecla']
eval_metrics = ['eval_micro_f1', 'eval_macro_f1']
labels = [
    'BRCA_NEGATIVO',
    'BRCA_POSITIVO',
    'CIRURGIA',
    'HER2_NEGATIVO',
    'HER2_POSITIVO',
    'POS_MENOPAUSA',
    'PRE_MENOPAUSA',
    'RE_NEGATIVO',
    'RE_POSITIVO',
    'RP_NEGATIVO',
    'RP_POSITIVO',
    'TIPO_HISTOPATOLOGICO',
]

rename_experiments = {
    'baseline': 'Baseline',
    'exp_1_mecla': 'BCE + MECLA',
    'exp_2_mecla': 'BCE + Pairwise MECLA',
    'exp_6_mecla': 'BCE + Grouped Softmax'
}

rename_metrics = {
    'eval_micro_f1': 'Micro F1',
    'eval_macro_f1': 'Macro F1',
    'BRCA_NEGATIVO_f1_score':'Negative BRCA - F1-score',
    'BRCA_POSITIVO_f1_score':'Positive BRCA - F1-score',
    'CIRURGIA_f1_score':'Surgery - F1-score',
    'HER2_NEGATIVO_f1_score':'Negative HER2 - F1-score',
    'HER2_POSITIVO_f1_score':'Positive HER2 - F1-score',
    'POS_MENOPAUSA_f1_score':'Post-Menopause - F1-score',
    'PRE_MENOPAUSA_f1_score':'Pre-Menopause - F1-score',
    'RE_NEGATIVO_f1_score':'Negative ER - F1-score',
    'RE_POSITIVO_f1_score':'Positive ER - F1-score',
    'RP_NEGATIVO_f1_score':'Negative PR - F1-score',
    'RP_POSITIVO_f1_score':'Positive PR - F1-score',
    'TIPO_HISTOPATOLOGICO_f1_score':'Histopathological type - F1-score',
}

include_label_specific_f1_scores = True

# ====== plotting config ======
OUTDIR = Path("plots/boxplots")
OUTDIR.mkdir(parents=True, exist_ok=True)

PLOT_GLOBAL_METRICS = True
PLOT_ENTITY_METRICS = True

# If True, one figure per metric (clean + paper-friendly).
# If False, group many metrics into pages (helpful when there are many entities).
ONE_FIGURE_PER_METRIC = True
METRICS_PER_FIG = 6  # only used when ONE_FIGURE_PER_METRIC=False

# ====== storage for stats tests etc ======
group_metrics_for_hypothesis_test = {}

# ====== helper ======
def _make_long_df(group_metrics_for_hypothesis_test, model: str, metrics: list[str]) -> pd.DataFrame:
    """
    Convert nested dict:
      group_metrics_for_hypothesis_test[model][exp][metric] -> list[float]
    into long DataFrame with columns: [model, metric, experiment, value].
    """
    rows = []
    for exp, metric_dict in group_metrics_for_hypothesis_test[model].items():
        exp_name = rename_experiments.get(exp, exp)
        for metric in metrics:
            if metric not in metric_dict:
                continue
            for v in metric_dict[metric]:
                rows.append(
                    {
                        "model": model,
                        "metric": metric,
                        "experiment": exp_name,
                        "value": float(v) if v is not None else None,
                    }
                )
    df_long = pd.DataFrame(rows).dropna(subset=["value"])
    return df_long


def _plot_boxplot_single(df_long: pd.DataFrame, model: str, metric: str, outpath: Path) -> None:
    """
    Make a single boxplot figure for one metric.
    Uses plain matplotlib (no seaborn) and no forced colors.
    """
    
    sub = df_long[(df_long["model"] == model) & (df_long["metric"] == metric)].copy()
    if sub.empty:
        print(f"[skip] No data for {model} / {metric}")
        return

    # preserve ordering from experiments list
    ordered_names = [rename_experiments.get(e, e) for e in experiments]
    sub["experiment"] = pd.Categorical(sub["experiment"], categories=ordered_names, ordered=True)
    sub = sub.sort_values("experiment")

    data = [sub.loc[sub["experiment"] == name, "value"].to_numpy() for name in ordered_names]

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.boxplot(
        data,
        tick_labels=ordered_names,
        showmeans=False,
        meanline=False,
    )
    ax.set_ylabel(rename_metrics[metric])
    ax.grid(True, axis="y", alpha=0.25)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)


def _plot_boxplots_paged(df_long: pd.DataFrame, model: str, metrics: list[str], outpath: Path) -> None:
    """
    Create a "page" (single figure) containing multiple boxplots stacked vertically.
    Still matplotlib-only, no seaborn. Good when you have many entity metrics.
    """
    sub = df_long[df_long["model"] == model].copy()
    ordered_names = [rename_experiments.get(e, e) for e in experiments]

    metrics = [m for m in metrics if ((sub["metric"] == m).any())]
    if not metrics:
        print(f"[skip] No metrics to plot for {model}")
        return

    n = len(metrics)
    fig_h = max(3.0, 2.2 * n)
    fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(8.5, fig_h), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        s = sub[sub["metric"] == metric].copy()
        s["experiment"] = pd.Categorical(s["experiment"], categories=ordered_names, ordered=True)
        s = s.sort_values("experiment")
        data = [s.loc[s["experiment"] == name, "value"].to_numpy() for name in ordered_names]

        ax.boxplot(
            data,
            labels=ordered_names,
            showmeans=True,
            meanline=False,
        )
        ax.set_ylabel(metric)
        ax.grid(True, axis="y", alpha=0.25)

    axes[0].set_title(f"{model} — boxplots")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)


# ====== main loop ======
for model in models:
    group_metrics_for_hypothesis_test[model] = {}

    # we’ll also collect which label cols exist in this run
    label_specific_f1_scores_df = None

    for exp in experiments:
        experiments_to_evaluate = list(Path(f'experiments/{exp}_{model}').glob('**/metrics_*.jsonl'))

        metrics = []
        for experiment_path in experiments_to_evaluate:
            with jsonlines.open(experiment_path) as reader:
                for obj in reader:
                    metrics.append(obj)

        df = pd.DataFrame(metrics)

        group_metrics_for_hypothesis_test[model][exp] = {}
        for metric in eval_metrics:
            group_metrics_for_hypothesis_test[model][exp][metric] = df[metric].to_list()

        if include_label_specific_f1_scores:
            if exp == 'exp_6_mecla':
                f1_score_per_label_cols = [col for col in df.columns if col.startswith('eval_f1_')]
                label_specific_f1_scores_df = df[f1_score_per_label_cols].copy()
                label_specific_f1_scores_df.columns = [
                    f"{col.split('eval_f1_')[1]}_f1_score" for col in label_specific_f1_scores_df.columns
                ]
                for col in label_specific_f1_scores_df.columns:
                    group_metrics_for_hypothesis_test[model][exp][col] = label_specific_f1_scores_df[col].to_list()
            else:
                label_specific_f1_scores = []
                for row in df.to_dict('records'):
                    label_specific_f1_score = {f"{k}_f1_score": v['f1_score'] for k, v in row['selected_thresholds'].items()}
                    label_specific_f1_scores.append(label_specific_f1_score)

                label_specific_f1_scores_df = pd.DataFrame(label_specific_f1_scores)
                for col in label_specific_f1_scores_df.columns:
                    group_metrics_for_hypothesis_test[model][exp][col] = label_specific_f1_scores_df[col].to_list()

        print(f'Experiment: {rename_experiments[exp]} - {model}')
        print('---')
        print('Global metrics:')
        print(f"Macro F1: {df['eval_macro_f1'].mean():.3f} ± {df['eval_macro_f1'].std():.3f}")
        print(f"Micro F1: {df['eval_micro_f1'].mean():.3f} ± {df['eval_micro_f1'].std():.3f}")
        print('---')
        if label_specific_f1_scores_df is not None:
            print('Per class metrics:')
            for col in label_specific_f1_scores_df.columns:
                print(f"{col:<30}: {label_specific_f1_scores_df[col].mean():.3f} ± {label_specific_f1_scores_df[col].std():.3f}")
        print('\n\n\n')

    # ====== plotting after we loaded all exps for this model ======
    # Decide which per-entity metric columns to plot
    entity_metric_names = [f"{lab}_f1_score" for lab in labels]

    metrics_to_plot = []
    if PLOT_GLOBAL_METRICS:
        metrics_to_plot += eval_metrics
    if PLOT_ENTITY_METRICS:
        metrics_to_plot += entity_metric_names

    df_long = _make_long_df(group_metrics_for_hypothesis_test, model=model, metrics=metrics_to_plot)

    if ONE_FIGURE_PER_METRIC:
        for metric in metrics_to_plot:
            outpath = OUTDIR / f"{model}__{metric}.png"
            _plot_boxplot_single(df_long, model=model, metric=metric, outpath=outpath)
        print(f"[done] Saved boxplots to: {OUTDIR.resolve()}")
    else:
        # global metrics in one page
        if PLOT_GLOBAL_METRICS:
            outpath = OUTDIR / f"{model}__global_metrics.png"
            _plot_boxplots_paged(df_long, model=model, metrics=eval_metrics, outpath=outpath)

        # entities as multiple pages
        if PLOT_ENTITY_METRICS:
            for i in range(0, len(entity_metric_names), METRICS_PER_FIG):
                chunk = entity_metric_names[i : i + METRICS_PER_FIG]
                outpath = OUTDIR / f"{model}__entity_metrics_{i//METRICS_PER_FIG+1:02d}.png"
                _plot_boxplots_paged(df_long, model=model, metrics=chunk, outpath=outpath)

        print(f"[done] Saved boxplots to: {OUTDIR.resolve()}")


Experiment: Baseline - mmbert
---
Global metrics:
Macro F1: 0.657 ± 0.035
Micro F1: 0.728 ± 0.037
---
Per class metrics:
BRCA_NEGATIVO_f1_score        : 0.458 ± 0.055
BRCA_POSITIVO_f1_score        : 0.382 ± 0.071
CIRURGIA_f1_score             : 0.836 ± 0.016
HER2_NEGATIVO_f1_score        : 0.801 ± 0.055
HER2_POSITIVO_f1_score        : 0.775 ± 0.063
POS_MENOPAUSA_f1_score        : 0.633 ± 0.035
PRE_MENOPAUSA_f1_score        : 0.370 ± 0.046
RE_NEGATIVO_f1_score          : 0.763 ± 0.038
RE_POSITIVO_f1_score          : 0.896 ± 0.034
RP_NEGATIVO_f1_score          : 0.767 ± 0.038
RP_POSITIVO_f1_score          : 0.884 ± 0.047
TIPO_HISTOPATOLOGICO_f1_score : 0.794 ± 0.024




Experiment: BCE + MECLA - mmbert
---
Global metrics:
Macro F1: 0.688 ± 0.023
Micro F1: 0.762 ± 0.028
---
Per class metrics:
BRCA_NEGATIVO_f1_score        : 0.459 ± 0.038
BRCA_POSITIVO_f1_score        : 0.430 ± 0.073
CIRURGIA_f1_score             : 0.842 ± 0.009
HER2_NEGATIVO_f1_score        : 0.853 ± 0.028
HER2_POSITIVO_f

In [38]:
from scipy import stats
from itertools import combinations
from cliffs_delta import cliffs_delta
from statsmodels.stats.multitest import multipletests

for model in models:
    for metric in eval_metrics:
        print(f'Model: {model}')
        print(f'Metric: {metric}')

        # Perform the Friedman test
        statistic, pvalue = stats.friedmanchisquare(
            *[
                group_metrics_for_hypothesis_test[model][exp][metric]
                for exp in experiments
            ]
        )

        print(f"Friedman test statistic: {statistic}")
        print(f"P-value: {pvalue:.5f}")
        print("=="*20)

        # Generate combinations of length 2
        combos_iterator = combinations(experiments, 2)

        # Convert the iterator to a list of tuples for display
        combos_list = list(combos_iterator)
        print('Combinations')
        print(combos_list)
        print("=="*20)
        # post hoc with Wilcoxon Signed-Rank Test for each pair and Cliff's Delta
        p_values = []
        test_results = []
        for group1, group2 in combos_list:
            stat, p = stats.wilcoxon(
                group_metrics_for_hypothesis_test[model][group1][metric],
                group_metrics_for_hypothesis_test[model][group2][metric]
                )
            p_values.append(p)
            test_results.append({'Comparison': f"{rename_experiments[group1]} vs {rename_experiments[group2]}", 'Raw P-Value': p})
            d, size = cliffs_delta(
                group_metrics_for_hypothesis_test[model][group1][metric], group_metrics_for_hypothesis_test[model][group2][metric]
                )
            
            print(f"Comparison: {rename_experiments[group1]} vs {rename_experiments[group2]}")
            print(f"Cliff's Delta: {d}")
            print(f"Effect size: {size}")
            print("=="*20)

        reject, corrected_p, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
        results_df = pd.DataFrame(test_results)
        results_df['Corrected P-Value'] = corrected_p
        results_df['Reject H0'] = reject
        print(results_df)

        print('\n\n')

Model: mmbert
Metric: eval_micro_f1
Friedman test statistic: 25.399999999999977
P-value: 0.00000
Combinations
[('baseline', 'exp_1_mecla'), ('baseline', 'exp_6_mecla'), ('exp_1_mecla', 'exp_6_mecla')]
Comparison: Baseline vs BCE + MECLA
Cliff's Delta: -0.5511111111111111
Effect size: large
Comparison: Baseline vs BCE + Grouped Softmax
Cliff's Delta: -0.8377777777777777
Effect size: large
Comparison: BCE + MECLA vs BCE + Grouped Softmax
Cliff's Delta: -0.5177777777777778
Effect size: large
                             Comparison   Raw P-Value  Corrected P-Value  \
0               Baseline vs BCE + MECLA  1.373943e-04       4.121829e-04   
1     Baseline vs BCE + Grouped Softmax  4.656613e-08       1.396984e-07   
2  BCE + MECLA vs BCE + Grouped Softmax  2.020182e-03       6.060546e-03   

   Reject H0  
0       True  
1       True  
2       True  



Model: mmbert
Metric: eval_macro_f1
Friedman test statistic: 22.399999999999977
P-value: 0.00001
Combinations
[('baseline', 'exp_1_mecla')

In [39]:
# Per label analysis
from scipy import stats
from itertools import combinations
from cliffs_delta import cliffs_delta
from statsmodels.stats.multitest import multipletests
import numpy as np

per_label_per_exp_f1_scores_and_pvalues = []

for model in models:
    for label in labels:
        metric = f"{label}_f1_score"
        print(f'Model: {model}')
        print(f'Metric: {metric}')

        current_label_values = {
            'Entity': label,
            }

        for exp in experiments:
            print(exp, len(group_metrics_for_hypothesis_test[model][exp][metric]))
            current_label_values[f'{rename_experiments[exp]}'] = f"{
                np.array(group_metrics_for_hypothesis_test[model][exp][metric]).mean():.3f} ± {
                    np.array(group_metrics_for_hypothesis_test[model][exp][metric]).std():.3f}"

        # Perform the Friedman test
        statistic, pvalue = stats.friedmanchisquare(
            *[
                group_metrics_for_hypothesis_test[model][exp][metric]
                for exp in experiments
            ]
        )

        print(f"Friedman test statistic: {statistic}")
        print(f"P-value: {pvalue:.5f}")
        print("=="*20)
        current_label_values['Friedman test statistic'] = f"{statistic:.1f}"
        current_label_values['P-value'] = f"{pvalue:.4f}" if pvalue >= 0.0001 else "<0.0001"

        per_label_per_exp_f1_scores_and_pvalues.append(current_label_values)

        # Generate combinations of length 2
        combos_iterator = combinations(experiments, 2)

        # Convert the iterator to a list of tuples for display
        combos_list = list(combos_iterator)
        print('Combinations')
        print(combos_list)
        print("=="*20)
        # post hoc with Wilcoxon Signed-Rank Test for each pair and Cliff's Delta
        p_values = []
        test_results = []
        for group1, group2 in combos_list:
            stat, p = stats.wilcoxon(
                group_metrics_for_hypothesis_test[model][group1][metric],
                group_metrics_for_hypothesis_test[model][group2][metric]
                )
            p_values.append(p)
            test_results.append({'Comparison': f"{rename_experiments[group1]} vs {rename_experiments[group2]}", 'Raw P-Value': p})
            d, size = cliffs_delta(
                group_metrics_for_hypothesis_test[model][group1][metric], group_metrics_for_hypothesis_test[model][group2][metric]
                )
            
            print(f"Comparison: {rename_experiments[group1]} vs {rename_experiments[group2]}")
            print(f"Cliff's Delta: {d}")
            print(f"Effect size: {size}")
            print("=="*20)

        reject, corrected_p, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
        results_df = pd.DataFrame(test_results)
        results_df['Corrected P-Value'] = corrected_p
        results_df['Reject H0'] = reject
        print(results_df)

        print('\n\n')

Model: mmbert
Metric: BRCA_NEGATIVO_f1_score
baseline 30
exp_1_mecla 30
exp_6_mecla 30
Friedman test statistic: 0.8666666666666742
P-value: 0.64834
Combinations
[('baseline', 'exp_1_mecla'), ('baseline', 'exp_6_mecla'), ('exp_1_mecla', 'exp_6_mecla')]
Comparison: Baseline vs BCE + MECLA
Cliff's Delta: -0.035555555555555556
Effect size: negligible
Comparison: Baseline vs BCE + Grouped Softmax
Cliff's Delta: 0.06444444444444444
Effect size: negligible
Comparison: BCE + MECLA vs BCE + Grouped Softmax
Cliff's Delta: 0.10222222222222223
Effect size: negligible
                             Comparison  Raw P-Value  Corrected P-Value  \
0               Baseline vs BCE + MECLA     0.700033                1.0   
1     Baseline vs BCE + Grouped Softmax     0.685047                1.0   
2  BCE + MECLA vs BCE + Grouped Softmax     0.640825                1.0   

   Reject H0  
0      False  
1      False  
2      False  



Model: mmbert
Metric: BRCA_POSITIVO_f1_score
baseline 30
exp_1_mecla 30
ex

In [41]:
pd.DataFrame(per_label_per_exp_f1_scores_and_pvalues)

,Entity,Baseline,BCE + MECLA,BCE + Grouped Softmax,Friedman test statistic,P-value
0,BRCA_NEGATIVO,0.458 ± 0.054,0.459 ± 0.037,0.453 ± 0.044,0.9,0.6483
1,BRCA_POSITIVO,0.382 ± 0.070,0.430 ± 0.071,0.400 ± 0.055,5.4,0.0672
2,CIRURGIA,0.836 ± 0.016,0.842 ± 0.009,0.840 ± 0.013,4.9,0.0877
3,HER2_NEGATIVO,0.801 ± 0.054,0.853 ± 0.028,0.807 ± 0.038,24.2,<0.0001
4,HER2_POSITIVO,0.775 ± 0.062,0.805 ± 0.022,0.777 ± 0.031,14.5,0.0007
5,POS_MENOPAUSA,0.633 ± 0.034,0.649 ± 0.034,0.638 ± 0.041,4.5,0.1072
6,PRE_MENOPAUSA,0.370 ± 0.046,0.394 ± 0.030,0.363 ± 0.056,7.2,0.0273
7,RE_NEGATIVO,0.763 ± 0.037,0.791 ± 0.042,0.734 ± 0.076,17.9,0.0001
8,RE_POSITIVO,0.896 ± 0.034,0.927 ± 0.011,0.902 ± 0.025,26.6,<0.0001
9,RP_NEGATIVO,0.767 ± 0.037,0.797 ± 0.028,0.765 ± 0.052,12.9,0.0016


In [43]:
pd.DataFrame(per_label_per_exp_f1_scores_and_pvalues).to_latex('per_label_f1_scores.tex', index=False)